# JDM NanoString: interferon score against the clinical picture
### Notebook `03_jdm_ifn_vs_clinical` (R). Self-contained: needs only the JDM workbook and the clinical database CSV.

**What this does.** Takes the raw sheet of `JDM IFN scores OADR.xlsx`, joins each sample to the clinical database on `nano ID`, computes the interferon score from the raw counts, and asks two questions inside the JDM set only: does the score track disease activity, and does it differ by myositis antibody?

**What it deliberately does not do.**
- It does not use the RGM set and does not pool anything across datasets. Notebook `02` was the trial of that and stays as a record; nothing here reads its output.
- It does not use the workbook's STD sheets. STD scores are defined against a fixed healthy-control reference whose per-gene mean and SD are not published, and this workbook has no healthy controls, so they cannot be recomputed. The notebook uses the geometric-mean score, which needs no reference.
- It does not find a "best" score. One score is chosen in advance in Part 1 (`PRIMARY_SCORE`).

| Part | Step |
|---|---|
| 1 | Settings |
| 2 | Load the raw sheet |
| 3 | Join to the clinical database |
| 4 | Compute the interferon score (and check it against the workbook's own) |
| 5 | One row per patient |
| 6 | Score against activity and labs |
| 7 | Score by antibody group |
| 8 | Repeat samples (optional) |
| 9 | Save |

**Privacy.** The clinical file and the saved outputs hold patient codes. Everything printed below is counts and summaries only. Keep the files on Lifebit and out of GitHub.

In [ ]:
library(readxl)

# ---- settings (change here, nowhere else) -----------------------------------
OUT    <- "/mnt/filesystems/nanostring_out"
SEARCH <- c("/mnt/filesystems", "/mnt/session_data", "/home/jovyan")
XLSX_REGEX <- "^JDM IFN scores OADR.*\\.xlsx$"
DB_REGEX   <- "^OADR-I-26-007.*clinical.*JDM.*\\.csv$"
XLSX <- NULL            # or a full path
DB   <- NULL            # or a full path
RAW_SHEET <- NULL       # NULL = pick the sheet named like "raw" (not "no form", not "STD"); or give its exact name

# Chosen BEFORE looking at any result. Either "geomean28" or "geomean25".
PRIMARY_SCORE <- "geomean28"

HK    <- c("ALAS1", "HPRT1", "TBP", "TUBB")
IFN28 <- c("CXCL10","DDX60","EPSTI1","GBP1","HERC5","HERC6","IFI27","IFI44","IFI44L","IFI6","IFIT1","IFIT2",
           "IFIT3","IFIT5","ISG15","LAMP3","LY6E","MX1","OAS1","OAS2","OAS3","OASL","RSAD2","RTP4","SIGLEC1",
           "SOCS1","SPATS2L","USP18")                              # Kim 2018, same as genes/ifn-response-genes-28.txt
IFN25 <- setdiff(IFN28, c("SIGLEC1", "SPATS2L", "EPSTI1"))        # the published 25-gene variant

# clinical columns pulled from the database (name in the file -> short name here)
CLIN <- c(MD_global = "MD_GLOBAL_ACT_ADJ_VAS", PT_global = "PT_GLOBAL_ACT_ADJ_VAS",
          DAS_total = "DAS_TOTAL_ADJ_SCORE", DAS_muscle = "DAS_MUSCLE_ADJ_SCORE", DAS_skin = "DAS_SKIN_ADJ_SCORE",
          MMT8 = "MMT_MMT8_ADJUSTED_SCORE",
          Aldolase = "Observation Value_Aldolase", AST = "Observation Value_AST", LDH = "Observation Value_LDH",
          CK = "Observation Value_CK", CRP = "Observation Value_CRP", ESR = "Observation Value_ESR",
          disease_duration = "disease_duration")
# CK is included on purpose: the published JDM work found it did NOT track the interferon score, so it is a sanity check.

find_one <- function(pat) {
  h <- unlist(lapply(SEARCH[dir.exists(SEARCH)], function(d)
    list.files(d, pattern = pat, recursive = TRUE, full.names = TRUE, ignore.case = TRUE)))
  h <- h[!grepl("~\\$", h)]
  if (length(h) == 0) stop("no file matching ", pat, " under ", paste(SEARCH, collapse = ", "))
  h[1]
}
if (is.null(XLSX)) XLSX <- find_one(XLSX_REGEX)
if (is.null(DB))   DB   <- find_one(DB_REGEX)
cat("workbook:", basename(XLSX), "\ndatabase:", basename(DB), "\n")
dir.create(OUT, recursive = TRUE, showWarnings = FALSE)

normalize_id <- function(x) {                     # O-6033 -> O6033, 015-01-01 -> 0150101, 1013.0 -> 1013
  s <- trimws(as.character(x)); s[is.na(x)] <- ""
  s[tolower(s) %in% c("nan", "none", "nat", "na")] <- ""
  toupper(gsub("[-_ ]", "", sub("\\.0$", "", s)))
}

## Part 2 · Load the raw sheet
🎯 **Goal:** read the `JDM - raw` sheet into a table with one row per NanoString sample, the 28 interferon genes and 4 housekeeping genes as numbers, and the sample's identifiers.
💡 **Why the care:** the header is not on row 1 (it was row 2 in your file), and the sheet holds blank rows. The header row is the one with the most exact gene names in it. Only exact gene names count as gene columns, so `CXCL10/IP-10` (a different measurement) is not mistaken for the CXCL10 gene.
👀 **Checkpoint:** the printed sheet name, header row, and row count. In your file there were 63 rows of which 2 were empty, so expect 61 samples.

In [ ]:
sheets <- excel_sheets(XLSX)
sh <- if (!is.null(RAW_SHEET)) RAW_SHEET else {
  s <- sheets[grepl("raw", sheets, ignore.case = TRUE) & !grepl("no|non|form|std", sheets, ignore.case = TRUE)]
  if (!length(s)) stop("no raw sheet found among: ", paste(sheets, collapse = " | "))
  s[1] }
cat("sheets:", paste(sheets, collapse = " | "), "\nusing :", sh, "\n")

raw <- as.data.frame(suppressMessages(read_excel(XLSX, sheet = sh, col_names = FALSE, col_types = "text")),
                     stringsAsFactors = FALSE)
raw <- raw[rowSums(!is.na(raw)) > 0, , drop = FALSE]
all_genes <- unique(c(IFN28, HK))
hits <- sapply(seq_len(min(15, nrow(raw))), function(i) sum(trimws(unlist(raw[i, ])) %in% all_genes))
h <- which.max(hits)
hdr <- trimws(unlist(raw[h, ])); bad <- is.na(hdr) | hdr == ""
hdr[bad] <- paste0("unnamed_", which(bad))
df <- raw[-seq_len(h), , drop = FALSE]
names(df) <- make.unique(hdr); rownames(df) <- NULL
cat("header row:", h, "| genes found in header:", hits[h], "of", length(all_genes), "\n")

need <- c("study sample", "Lab initials", "sample date")
if (!all(need %in% names(df))) stop("expected columns missing: ", paste(setdiff(need, names(df)), collapse = ", "),
                                    "\nfound: ", paste(names(df), collapse = " | "))
empty <- is.na(df[["study sample"]]) & is.na(df[["Lab initials"]])
cat("empty rows dropped:", sum(empty), "| samples kept:", sum(!empty), "\n")
df <- df[!empty, , drop = FALSE]
miss <- setdiff(all_genes, names(df))
if (length(miss)) stop("genes missing from the sheet: ", paste(miss, collapse = ", "))
for (g in all_genes) df[[g]] <- suppressWarnings(as.numeric(df[[g]]))
cat("rows with any missing gene count:", sum(!complete.cases(df[, all_genes])), "\n")

## Part 3 · Join to the clinical database
🎯 **Goal:** attach each sample's clinical row (activity scores, labs, antibodies, visit date) from the database CSV.
💡 **Why `nano ID`:** it is the database's own key to this workbook. In your data 55 of the 61 samples matched it (51 on the study code, 4 on the lab-initials column). Earlier tries matched on initials against the RNA-seq list and found only 27 of 119, so that route is not used. Samples with no database row are kept and labelled `no_clinical_record`; they get a score but no clinical values.
👀 **Checkpoint:** the match table. A large "no_clinical_record" count means the key is written differently in the two files. One `nano ID` appears twice in the database on different visit dates; the first row is used, which can mislabel at most that one sample.

In [ ]:
db <- read.csv(DB, check.names = FALSE, stringsAsFactors = FALSE, colClasses = "character")
if (!"nano ID" %in% names(db)) stop("database has no 'nano ID' column")
db$.key <- normalize_id(db[["nano ID"]])
cat("database:", nrow(db), "rows | duplicated nano IDs:", sum(duplicated(db$.key[db$.key != ""])), "\n")
d1 <- db[db$.key != "" & !duplicated(db$.key), , drop = FALSE]        # first row per nano ID

keys <- list(`study sample` = normalize_id(df[["study sample"]]),
             `Lab initials` = normalize_id(df[["Lab initials"]]),
             `nano ID`      = if ("nano ID" %in% names(df)) normalize_id(df[["nano ID"]]) else rep("", nrow(df)))
mi <- rep(NA_integer_, nrow(df)); route <- rep("no_clinical_record", nrow(df))
for (k in names(keys)) {
  m <- match(keys[[k]], d1$.key); use <- is.na(mi) & !is.na(m)
  mi[use] <- m[use]; route[use] <- k
}
df$db_route <- route
cat("\nhow samples matched the database:\n"); print(table(df$db_route))

pick <- function(nm) { i <- which(names(d1) == nm)[1]; if (is.na(i)) rep(NA_character_, nrow(df)) else d1[[i]][mi] }
num  <- function(x) suppressWarnings(as.numeric(x))
for (nm in names(CLIN)) df[[nm]] <- num(pick(CLIN[[nm]]))
for (ab in c("P155", "MDA5", "MJ")) df[[paste0("ab_", ab)]] <- pick(ab)
df$MSA_basic <- pick("Basic_MSA")
df$dx        <- pick("DX_PROTOCOL")
cat("\ndiagnosis (DX_PROTOCOL) among matched samples:\n"); print(table(df$dx, useNA = "ifany"))
cat("\nclinical columns, non-missing samples:\n")
print(sapply(c(names(CLIN), "ab_P155", "ab_MDA5", "ab_MJ"), function(cn) sum(!is.na(df[[cn]]) & df[[cn]] != "")))

## Part 4 · The interferon score
🎯 **Goal:** one number per sample that measures how strongly the 28 interferon genes are on.
💡 **How (the published geometric-mean score, Kim 2018):** scale each sample so its four housekeeping genes (ALAS1, HPRT1, TBP, TUBB) have a geometric mean of 1000, then take the geometric mean of the 28 interferon genes' counts and divide by 10. It needs no healthy controls. `geomean25` drops SIGLEC1, SPATS2L and EPSTI1. Two things differ from the published pipeline: positive-control scaling is skipped (the workbook has no positive-control probes), and counts of 0 are floored at 0.5 so the log is defined.
👀 **Checkpoint:** the workbook already has its own `Geomean 28` and `Geomean 25` columns. Our recomputed score should rank samples almost identically (Spearman near 1). If it does not, the raw sheet is not what I assumed and the rest should not be trusted.

In [ ]:
cnt   <- as.matrix(df[, all_genes]); storage.mode(cnt) <- "double"
hkg   <- exp(rowMeans(log(pmax(cnt[, HK], 0.5))))
cn    <- cnt / hkg * 1000                                    # housekeeping geomean = 1000 per sample
gm    <- function(genes) exp(rowMeans(log(pmax(cn[, genes], 0.5)))) / 10
df$geomean28 <- gm(IFN28); df$geomean25 <- gm(IFN25)
df$score <- df[[PRIMARY_SCORE]]
cat("primary score:", PRIMARY_SCORE, "| range:", paste(round(range(df$score, na.rm = TRUE), 1), collapse = " to "), "\n\n")

cat("agreement with the workbook's own score (Spearman):\n")
for (pair in list(c("geomean28", "Geomean 28"), c("geomean25", "Geomean 25"))) {
  if (pair[2] %in% names(df)) {
    w <- num(df[[pair[2]]])
    cat(sprintf("  %-10s vs '%s': rho = %.3f (n = %d)\n", pair[1], pair[2],
                suppressWarnings(cor(df[[pair[1]]], w, method = "spearman", use = "complete.obs")), sum(!is.na(w))))
  } else cat("  workbook column", pair[2], "not found\n")
}

## Part 5 · One row per patient
🎯 **Goal:** a table with each patient counted once (their earliest sample), used for every test below.
💡 **Why:** the raw sheet has about 61 samples from about 49 patients. Some patients have several samples, and treating those as independent makes correlations look stronger than they are. In your data the activity correlation fell from about 0.25 across all samples to about 0.1 once each patient was counted once.
👀 **Checkpoint:** samples vs patients.

In [ ]:
df$patient <- sub("-.*$", "", trimws(df[["study sample"]]))
df$sample_date <- as.Date(num(df[["sample date"]]), origin = "1899-12-30")      # Excel serial -> date
df <- df[order(df$patient, df$sample_date), ]
p1 <- df[!duplicated(df$patient), ]
cat("samples:", nrow(df), "| patients:", nrow(p1), "| patients with a clinical record:", sum(p1$db_route != "no_clinical_record"), "\n")
cat("sample dates:", format(min(df$sample_date, na.rm = TRUE)), "to", format(max(df$sample_date, na.rm = TRUE)), "\n")

## Part 6 · Does the score track activity and labs?
🎯 **Goal:** the rank correlation (Spearman) of the interferon score with each clinical measure, one row per patient.
💡 **Reading it:** with about 40 patients only a correlation of roughly 0.3 or more can stand out from chance, so a small rho means "not detectable here", not "no relationship". Seven or more measures are tested, so the adjusted p-value (Benjamini-Hochberg) is the one to look at. `CK` is a built-in sanity check: published JDM work found it did not track the score.
👀 **Checkpoint:** the table, sorted by p. In the earlier trial the physician global score gave rho about 0.1 (n = 42).

In [ ]:
res <- do.call(rbind, lapply(names(CLIN), function(v) {
  ok <- complete.cases(p1$score, p1[[v]])
  if (sum(ok) < 8) return(data.frame(measure = v, n = sum(ok), rho = NA, p = NA))
  ct <- suppressWarnings(cor.test(p1$score[ok], p1[[v]][ok], method = "spearman", exact = FALSE))
  data.frame(measure = v, n = sum(ok), rho = round(unname(ct$estimate), 2), p = signif(ct$p.value, 2)) }))
res$p_BH <- signif(p.adjust(res$p, "BH"), 2)
print(res[order(res$p), ], row.names = FALSE)

if ("MD_global" %in% names(p1)) {
  plot(p1$MD_global, p1$score, pch = 19, log = "y", xlab = "physician global activity", ylab = paste(PRIMARY_SCORE, "(log scale)"),
       main = "Interferon score vs physician global (one point per patient)")
}

## Part 7 · Does the score differ by antibody group?
🎯 **Goal:** compare the score across anti-MDA5, anti-P155 (usually TIF1γ), anti-MJ (usually NXP2), and patients negative for all three.
💡 **Why four groups and not three yes/no tests:** these antibodies almost never occur together, so "MDA5-negative" mixes in the high-scoring P155 and MDA5 patients' neighbours and the MJ patients' comparison group looks artificially high. Here each patient is in exactly one group and every group is compared with the all-negative group, with a Holm correction. Patients positive for more than one antibody are listed as `multiple` and left out of the tests.
👀 **Checkpoint:** the group sizes. A group under about 6 patients cannot carry a conclusion. In the trial, anti-MDA5 had the highest median (9 patients) but the adjusted p-value against the all-negative group was 0.45, so it counted as a hint only.

In [ ]:
pos <- function(x) as.integer(x %in% "POS")
p1$npos <- pos(p1$ab_P155) + pos(p1$ab_MDA5) + pos(p1$ab_MJ)
allneg  <- p1$ab_P155 %in% "NEG" & p1$ab_MDA5 %in% "NEG" & p1$ab_MJ %in% "NEG"
p1$abgroup <- ifelse(p1$npos > 1, "multiple",
              ifelse(p1$ab_MDA5 %in% "POS", "MDA5", ifelse(p1$ab_P155 %in% "POS", "P155",
              ifelse(p1$ab_MJ %in% "POS", "MJ", ifelse(allneg, "none_of_3", "unknown")))))
cat("patients by antibody group:\n"); print(table(p1$abgroup))

d <- p1[!is.na(p1$score) & p1$abgroup %in% c("none_of_3", "P155", "MJ", "MDA5"), ]
d$abgroup <- factor(d$abgroup, levels = c("none_of_3", "P155", "MJ", "MDA5"))
tab <- do.call(rbind, lapply(levels(d$abgroup), function(g) { x <- d[d$abgroup == g, ]
  data.frame(group = g, n = nrow(x), median_score = round(median(x$score), 1),
             median_sample_year = median(as.numeric(format(x$sample_date, "%Y")), na.rm = TRUE)) }))
print(tab, row.names = FALSE)
if (nlevels(droplevels(d$abgroup)) > 1) {
  cat("\nKruskal-Wallis p:", signif(kruskal.test(score ~ abgroup, data = d)$p.value, 2), "\n")
  cat("pairwise Wilcoxon, Holm-adjusted p:\n")
  print(signif(pairwise.wilcox.test(d$score, d$abgroup, p.adjust.method = "holm", exact = FALSE)$p.value, 2))
  boxplot(score ~ abgroup, data = d, las = 1, log = "y", ylab = paste(PRIMARY_SCORE, "(log scale)"), main = "Interferon score by antibody group")
  stripchart(score ~ abgroup, data = d, vertical = TRUE, method = "jitter", pch = 19, add = TRUE)
}

# does sampling date matter? (trial result: no, rho about 0)
ok <- !is.na(p1$score) & !is.na(p1$sample_date)
ct <- suppressWarnings(cor.test(p1$score[ok], as.numeric(p1$sample_date[ok]), method = "spearman", exact = FALSE))
cat("\nscore vs sample date, Spearman rho =", round(unname(ct$estimate), 2), "| p =", signif(ct$p.value, 2), "| n =", sum(ok), "\n")

## Part 8 · Repeat samples (optional)
🎯 **Goal:** use every sample, not only each patient's first, while accounting for patients having several.
💡 **How:** a mixed model of log(score) on physician global activity with a random intercept per patient (the approach used in the published JDM interferon-score papers). It needs the `nlme` package; if it is not installed this part prints a message and is skipped, and nothing else depends on it.

In [ ]:
if (requireNamespace("nlme", quietly = TRUE) && "MD_global" %in% names(df)) {
  m <- df[!is.na(df$score) & !is.na(df$MD_global), ]
  m$lg <- log(m$score)
  cat("samples:", nrow(m), "| patients:", length(unique(m$patient)), "| patients with more than one sample:", sum(table(m$patient) > 1), "\n")
  fit <- nlme::lme(lg ~ MD_global, random = ~ 1 | patient, data = m)
  print(round(summary(fit)$tTable, 3))
  cat("MD_global coefficient is the change in log(score) per physician-global point.\n")
} else cat("nlme not installed or no activity column; skipped.\n")

## Part 9 · Save
Two files in `OUT`, both with patient codes, so keep them on Lifebit: one row per sample (`jdm_ifn_samples.csv`) and one row per patient (`jdm_ifn_patients.csv`).

## What this notebook can and cannot show
* **Can:** whether, inside JDM, the geometric-mean interferon score moves with activity, labs and antibody group, with the published method.
* **Cannot:** compare JDM with healthy people or with other diseases (the workbook has no healthy controls, and the RGM set is a different set of patients), or reproduce the STD score (its healthy-control reference is not published).
* **Next, outside this notebook:** the same patients have RNA-seq (`RNA_Seq ID` in the database). Scoring them on the RNA-seq side and comparing with this score is the independent check on any antibody result.

In [ ]:
keep <- intersect(c("patient", "sample_date", "db_route", "dx", "MSA_basic", "ab_P155", "ab_MDA5", "ab_MJ",
                    "geomean28", "geomean25", "score", names(CLIN)), names(df))
write.csv(df[, keep], file.path(OUT, "jdm_ifn_samples.csv"), row.names = FALSE, na = "")
write.csv(p1[, intersect(c(keep, "abgroup"), names(p1))], file.path(OUT, "jdm_ifn_patients.csv"), row.names = FALSE, na = "")
cat("saved jdm_ifn_samples.csv and jdm_ifn_patients.csv in", OUT, "\n")